# 01 — Exploración de Datos · Flight Price Predictor
**Proyecto 205.01 · Paso 1: ML Clásico**

Este notebook tiene dos tipos de celdas:
- ✅ **EJEMPLO** — el código ya está escrito, correlo y entendé qué hace
- 🔍 **DESAFÍO** — hay una pista pero tenés que completar el código vos

> Antes de empezar: asegurate de haber corrido `python generate_data.py`


---
## Sección 1 — Cargar y entender los datos


### ✅ EJEMPLO 1.1 — Cargar el CSV y ver las primeras filas

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Configuración visual
plt.rcParams["figure.figsize"] = (10, 5)
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False
sns.set_palette("husl")

# Cargamos el dataset
df = pd.read_csv("../data/flights_synthetic.csv")

# .head() muestra las primeras 5 filas — siempre el primer paso
df.head()


**¿Qué estás viendo?**
Cada fila es un vuelo. Las columnas son las *features* (variables de entrada) más el *target* (`price_usd`).
Notá que `origin` siempre es Buenos Aires — en este dataset simulamos rutas desde Argentina.


### ✅ EJEMPLO 1.2 — Forma del dataset y tipos de datos

In [ ]:
# .shape devuelve (filas, columnas)
print(f"Filas:    {df.shape[0]:,}")
print(f"Columnas: {df.shape[1]}")
print()

# .dtypes muestra el tipo de cada columna
# object = texto (categórico), int64/float64 = número
print(df.dtypes)


**¿Por qué importa el tipo de dato?**
Los modelos ML solo entienden números. Las columnas `object` (como `airline` o `day_of_week`)
van a necesitar transformación antes de entrar al modelo — eso lo hace el `OneHotEncoder`.


### ✅ EJEMPLO 1.3 — Estadísticas descriptivas del target

In [ ]:
# .describe() calcula count, mean, std, min, percentiles, max
# Lo aplicamos solo al precio porque es lo que queremos predecir
df["price_usd"].describe().round(2)


**Leé esto con cuidado:**
- `mean` — precio promedio. ¿Tiene sentido para vuelos desde Argentina?
- `std` — desviación estándar. Alta variación = el modelo tiene mucho que aprender
- `min` / `max` — ¿hay precios que parecen imposibles?


### 🔍 DESAFÍO 1.4 — ¿Cuántos valores únicos tiene cada columna categórica?

**Pista:** usá `.nunique()` sobre el DataFrame filtrado con solo las columnas categóricas.
Las columnas categóricas son: `origin`, `destination`, `airline`, `day_of_week`.

**Lo que esperás ver:** destination debería tener 8 valores únicos, airline 4, day_of_week 7.


In [ ]:
# TU CÓDIGO ACÁ
# Pista: df[lista_de_columnas].nunique()

categorical_cols = ["origin", "destination", "airline", "day_of_week"]

# ¿Cuántos valores únicos tiene cada una?



### 🔍 DESAFÍO 1.5 — ¿Hay valores nulos en el dataset?

**Pista:** `.isnull().sum()` cuenta los nulos por columna.

**Lo que esperás ver:** todos en 0 — los datos sintéticos son limpios, pero en datos reales
los nulos son uno de los problemas más comunes y hay que decidir cómo tratarlos.


In [ ]:
# TU CÓDIGO ACÁ
# ¿Tiene nulos el dataset?



---
## Sección 2 — Distribución del precio (el target)

Antes de entrenar cualquier modelo, entendé bien qué es lo que querés predecir.


### ✅ EJEMPLO 2.1 — Histograma del precio

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Histograma simple
axes[0].hist(df["price_usd"], bins=50, color="#1E88E5", edgecolor="white", linewidth=0.5)
axes[0].set_xlabel("Precio (USD)")
axes[0].set_ylabel("Cantidad de vuelos")
axes[0].set_title("Distribución del precio")

# KDE (Kernel Density Estimate) — versión suavizada del histograma
# Muestra la forma de la distribución más claramente
df["price_usd"].plot(kind="kde", ax=axes[1], color="#1E88E5", linewidth=2)
axes[1].set_xlabel("Precio (USD)")
axes[1].set_title("Densidad del precio (KDE)")

plt.tight_layout()
plt.show()

print(f"Precio mediano:  ${df['price_usd'].median():.2f}")
print(f"Precio promedio: ${df['price_usd'].mean():.2f}")
print(f"¿La distribución es simétrica o tiene cola a la derecha?")


**¿Qué buscás ver?**
Si la distribución tiene una cola larga hacia la derecha (precios muy altos), significa que
hay vuelos outliers que pueden dificultar el entrenamiento. El modelo puede "distraerse"
tratando de predecir esos casos extremos.


### 🔍 DESAFÍO 2.2 — Boxplot del precio por aerolínea

**Pista:** `sns.boxplot(data=df, x="airline", y="price_usd")`

**Lo que esperás ver:** Aerolínea A debería tener precios más altos (es premium +15%).
Aerolínea C y D deberían ser más baratas (low cost -20%).

**Pregunta para reflexionar:** ¿Por qué hay vuelos de Aerolínea C que son más caros
que algunos de Aerolínea A? Pensá en qué otras variables influyen en el precio.


In [ ]:
# TU CÓDIGO ACÁ
# Boxplot de precio por aerolínea
# Pista: un boxplot muestra mediana, Q1, Q3 e outliers



### 🔍 DESAFÍO 2.3 — Precio promedio por día de la semana

**Pista:** usá `.groupby("day_of_week")["price_usd"].mean()` y después `.plot(kind="bar")`.

**Lo que esperás ver:** Viernes y Domingo deberían ser los más caros (+20%).
Martes y Miércoles deberían ser los más baratos (-10%).

Si no ves ese patrón claramente, ¿por qué puede ser? Pensá en cuántas otras
variables también afectan el precio.


In [ ]:
# TU CÓDIGO ACÁ
# Precio promedio por día de la semana, ordenado de mayor a menor



---
## Sección 3 — Relación entre features y precio

Acá analizamos si cada variable realmente "explica" parte del precio.
Esto se llama análisis de correlación y es clave para entender qué features
va a encontrar útiles el modelo.


### ✅ EJEMPLO 3.1 — Correlación entre variables numéricas

In [ ]:
# La correlación mide la relación lineal entre dos variables
# Va de -1 (relación inversa perfecta) a +1 (relación directa perfecta)
# 0 = no hay relación lineal (pero puede haber relación no lineal)

numerical_cols = ["distance_km", "days_in_advance", "departure_hour", "num_stops", "price_usd"]
corr_matrix = df[numerical_cols].corr()

plt.figure(figsize=(7, 5))
sns.heatmap(
    corr_matrix,
    annot=True,          # muestra los valores dentro de las celdas
    fmt=".2f",           # 2 decimales
    cmap="coolwarm",     # azul = negativo, rojo = positivo
    center=0,
    linewidths=0.5,
)
plt.title("Matriz de correlación — variables numéricas")
plt.tight_layout()
plt.show()


**¿Qué buscás ver?**
- `distance_km` vs `price_usd`: correlación alta positiva — más distancia, más caro
- `days_in_advance` vs `price_usd`: correlación negativa — más anticipación, más barato
- `num_stops` vs `price_usd`: correlación negativa leve — más escalas, más barato

**Ojo:** correlación baja no significa que la variable no sea útil.
`departure_hour` puede tener baja correlación lineal pero ser muy útil para el Random Forest
porque la relación es no lineal (madrugada barata, hora pico cara).


### ✅ EJEMPLO 3.2 — Precio vs días de anticipación (scatter plot)

In [ ]:
# Tomamos una muestra de 800 puntos para que el gráfico no sea muy denso
sample = df.sample(800, random_state=42)

plt.figure(figsize=(10, 5))
scatter = plt.scatter(
    sample["days_in_advance"],
    sample["price_usd"],
    c=sample["distance_km"],   # color según distancia
    cmap="viridis",
    alpha=0.6,
    s=20,
)
plt.colorbar(scatter, label="Distancia (km)")
plt.xlabel("Días de anticipación")
plt.ylabel("Precio (USD)")
plt.title("Precio vs anticipación — color = distancia del vuelo")
plt.show()

print("Observá:")
print("- ¿Se nota que comprar muy tarde (1-7 días) es más caro?")
print("- ¿Los puntos más amarillos (vuelos largos) tienden a estar más arriba?")


### 🔍 DESAFÍO 3.3 — Precio vs distancia con línea de tendencia

**Pista:** `sns.regplot(data=df, x="distance_km", y="price_usd", scatter_kws={"alpha": 0.2, "s": 5})`

`regplot` dibuja el scatter Y una línea de regresión con banda de confianza.

**Lo que esperás ver:** una tendencia positiva clara — más distancia, más caro.
Pero notarás que hay mucha dispersión: vuelos a la misma distancia tienen precios
muy distintos. ¿Qué otras variables explican esa dispersión?


In [ ]:
# TU CÓDIGO ACÁ
# Scatter con línea de tendencia: precio vs distancia



### 🔍 DESAFÍO 3.4 — ¿La hora de salida afecta el precio?

**Pista:** calculá el precio promedio por hora con `.groupby("departure_hour")["price_usd"].mean()`
y graficalo con `.plot()`.

**Lo que esperás ver:** una curva con:
- Horas 0-6 (madrugada): precios bajos
- Horas 7-9 y 17-20 (hora pico): precios altos
- El resto: precios intermedios

**Si no ves ese patrón perfectamente:** es normal. El ruido aleatorio y las otras
variables "tapan" un poco el efecto de la hora. Esto es por qué necesitamos ML
en vez de reglas simples.


In [ ]:
# TU CÓDIGO ACÁ
# Precio promedio por hora de salida — ¿se notan las horas pico?



---
## Sección 4 — Análisis por ruta

Las rutas tienen mucho que ver con el precio — no solo por la distancia,
sino por la demanda y la competencia entre aerolíneas.


### ✅ EJEMPLO 4.1 — Precio promedio por destino

In [ ]:
# Calculamos estadísticas por destino
route_stats = df.groupby("destination")["price_usd"].agg(["mean", "std", "count"])
route_stats.columns = ["precio_promedio", "desviacion", "vuelos"]
route_stats = route_stats.sort_values("precio_promedio", ascending=False)

print(route_stats.round(2))
print()

# Gráfico de barras horizontales con barras de error
fig, ax = plt.subplots(figsize=(10, 5))
ax.barh(
    route_stats.index,
    route_stats["precio_promedio"],
    xerr=route_stats["desviacion"],   # barras de error = desviación estándar
    color="#1E88E5",
    alpha=0.8,
    capsize=4,
    ecolor="gray",
)
ax.set_xlabel("Precio promedio (USD)")
ax.set_title("Precio promedio por destino ± desviación estándar")
plt.tight_layout()
plt.show()


**¿Qué significan las barras de error?**
La longitud de cada barra de error muestra la desviación estándar — cuánto varían
los precios alrededor del promedio. Destinos con barras largas tienen precios muy
variables (difíciles de predecir). Destinos con barras cortas son más predecibles.


### 🔍 DESAFÍO 4.2 — ¿Qué aerolínea domina cada ruta?

**Pista:** usá `pd.crosstab(df["destination"], df["airline"])` para ver cuántos
vuelos de cada aerolínea hay por destino.

**Lo que esperás ver:** una tabla donde podés comparar la presencia de cada aerolínea
en cada destino. En datos reales esto afecta mucho el precio (más competencia = más barato).

**Bonus:** si querés, podés normalizar con `normalize="index"` para ver porcentajes en vez de conteos.


In [ ]:
# TU CÓDIGO ACÁ
# Tabla cruzada: destino x aerolínea



---
## Sección 5 — Preparación mental para el modelo

Esta sección no tiene código — son reflexiones que deben guiar tus decisiones
antes de entrenar.


### ✅ EJEMPLO 5.1 — ¿Qué features deberías incluir?

Después de explorar los datos, podés clasificar las features por su utilidad esperada:


In [ ]:
# Este no es código ejecutable — es un resumen para pensar
resumen = {
    "Feature": [
        "distance_km",
        "days_in_advance",
        "airline",
        "day_of_week",
        "departure_hour",
        "num_stops",
        "origin",
    ],
    "Tipo": [
        "Numérica",
        "Numérica",
        "Categórica",
        "Categórica",
        "Numérica",
        "Numérica",
        "Categórica",
    ],
    "Impacto esperado": [
        "MUY ALTO — correlación directa con precio",
        "ALTO — comprar tarde es mucho más caro",
        "ALTO — premium vs low cost difiere ~35%",
        "MEDIO — viernes/domingo +20%, martes -10%",
        "MEDIO — madrugada barata, hora pico cara",
        "BAJO — cada escala baja ~$25",
        "NINGUNO — siempre Buenos Aires, no aporta info",
    ]
}

import pandas as pd
pd.DataFrame(resumen)


**Nota sobre `origin`:** aunque la incluimos en el modelo (para que el pipeline
sea genérico), en la práctica no aporta información — todos los vuelos salen de Buenos Aires.
El modelo lo va a aprender y le va a dar importancia ≈ 0.


### 🔍 DESAFÍO 5.2 — Detectá un posible outlier

**Contexto:** en el gráfico 3.2 probablemente viste algunos puntos muy por encima
de los demás. Esos son *outliers* — valores extremos que pueden ser errores o casos reales.

**Pista:** filtrá los vuelos con `price_usd > 1500` y analizá qué tienen en común.

**Preguntas para responder:**
1. ¿Son vuelos largos o cortos?
2. ¿Se compraron con poca anticipación?
3. ¿Son de aerolínea premium?
4. ¿Viajan un día de alta demanda?

**Reflexión final:** ¿deberías eliminar esos outliers antes de entrenar, o dejarlos?
No hay una sola respuesta correcta — depende de si representan casos reales de negocio.


In [ ]:
# TU CÓDIGO ACÁ
# Analizá los vuelos más caros (price_usd > 1500)
# ¿Qué tienen en común?



---
## ¿Qué sigue?

Con este análisis exploratorio ya tenés intuición sobre los datos.
El próximo paso es el entrenamiento formal en `train_model.py`:

```bash
python train_model.py
```

Y después visualizás las predicciones en vivo:

```bash
streamlit run app/streamlit_app.py
```

→ **Paso 2:** [205.02 - Document Chatbot](../205.02-document-chatbot) · LLMs + RAG
